# 01. Single-Experiment Registry Walkthrough

## Purpose
This notebook is the canonical first walkthrough for the registry-driven path.
It demonstrates one predefined experiment from the shipped registry through:
1. safe dry-run planning (default),
2. optional full execution,
3. output inspection,
4. artifact provenance inspection.

## Prerequisites
- Repository checkout with environment prepared (for example: `uv sync --frozen --extra dev`)
- A valid dataset index CSV and data root for execution steps
- CLI access to `thesisml-run-decision-support`

## Expected runtime
- Health check + dry-run: usually short
- Full execution: data-dependent and potentially much longer

## Expected outputs
Notebook-local artifacts under:
`outputs/notebook_demo/01_single_experiment/`

## Not covered here
- Workbook-driven studies
- Designed/factorial workflows
- Inferential scientific claims


## Configuration (edit this cell)

Set runtime flags and optional path overrides here.
Defaults are portable and repo-relative where possible.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from Thesis_ML.demo.notebook_utils import (
    discover_project_root,
    path_status,
    resolve_cli_command,
    resolve_user_path,
    setup_notebook_workspace,
)


# =========================
# USER-EDITABLE FLAGS
# =========================
EXPERIMENT_ID = os.getenv("THESIS_ML_NOTEBOOK_EXPERIMENT_ID", "E02").strip() or "E02"
RUN_HEALTH_CHECK = True
RUN_DRY_RUN = True
RUN_FULL_EXECUTION = False  # keep False for the first walkthrough


# =========================
# OPTIONAL PATH OVERRIDES
# =========================
INDEX_CSV_OVERRIDE = os.getenv("THESIS_ML_INDEX_CSV", "").strip()
DATA_ROOT_OVERRIDE = os.getenv("THESIS_ML_DATA_ROOT", "").strip()
REGISTRY_PATH_OVERRIDE = os.getenv("THESIS_ML_REGISTRY_PATH", "").strip()


PROJECT_ROOT = discover_project_root()
WORKSPACE = setup_notebook_workspace("01_single_experiment", project_root=PROJECT_ROOT)

INDEX_CSV = resolve_user_path(
    INDEX_CSV_OVERRIDE,
    base=PROJECT_ROOT,
    default="Data/processed/dataset_index.csv",
)
DATA_ROOT = resolve_user_path(
    DATA_ROOT_OVERRIDE,
    base=PROJECT_ROOT,
    default="Data",
)
REGISTRY_PATH = resolve_user_path(
    REGISTRY_PATH_OVERRIDE,
    base=PROJECT_ROOT,
    default="configs/decision_support_registry.json",
)

NOTEBOOK_ROOT = WORKSPACE.notebook_root
CACHE_DIR = WORKSPACE.cache_dir
OUTPUT_ROOT = WORKSPACE.output_root
NOTEBOOK_REGISTRY_DIR = WORKSPACE.registry_dir

DECISION_SUPPORT_CMD = resolve_cli_command(
    preferred_executable="thesisml-run-decision-support",
    fallback_module="Thesis_ML.cli.decision_support",
)

statuses = path_status(
    {
        "PROJECT_ROOT": PROJECT_ROOT,
        "INDEX_CSV": INDEX_CSV,
        "DATA_ROOT": DATA_ROOT,
        "REGISTRY_PATH": REGISTRY_PATH,
        "CACHE_DIR": CACHE_DIR,
        "OUTPUT_ROOT": OUTPUT_ROOT,
        "NOTEBOOK_REGISTRY_DIR": NOTEBOOK_REGISTRY_DIR,
    }
)

settings_rows = [
    ("PROJECT_ROOT", statuses["PROJECT_ROOT"]["path"]),
    ("EXPERIMENT_ID", EXPERIMENT_ID),
    ("RUN_HEALTH_CHECK", RUN_HEALTH_CHECK),
    ("RUN_DRY_RUN", RUN_DRY_RUN),
    ("RUN_FULL_EXECUTION", RUN_FULL_EXECUTION),
    ("INDEX_CSV", statuses["INDEX_CSV"]["path"]),
    ("INDEX_CSV_EXISTS", statuses["INDEX_CSV"]["exists"]),
    ("DATA_ROOT", statuses["DATA_ROOT"]["path"]),
    ("DATA_ROOT_EXISTS", statuses["DATA_ROOT"]["exists"]),
    ("REGISTRY_PATH", statuses["REGISTRY_PATH"]["path"]),
    ("REGISTRY_PATH_EXISTS", statuses["REGISTRY_PATH"]["exists"]),
    ("CACHE_DIR", statuses["CACHE_DIR"]["path"]),
    ("OUTPUT_ROOT", statuses["OUTPUT_ROOT"]["path"]),
    ("NOTEBOOK_REGISTRY_DIR", statuses["NOTEBOOK_REGISTRY_DIR"]["path"]),
]
settings = pd.DataFrame(settings_rows, columns=["setting", "value"])
display(settings)
print("Decision-support command prefix:", " ".join(DECISION_SUPPORT_CMD))


In [ ]:
from Thesis_ML.demo.notebook_utils import (
    dedupe_paths,
    list_campaign_manifests,
    preview_csv_rows,
    print_tree,
    read_json_safe,
    run_command,
)


def run(cmd: list[str], *, cwd: Path | None = None, check: bool = False):
    return run_command(cmd, cwd=cwd or PROJECT_ROOT, check=check)


def safe_read_json(path: Path):
    return read_json_safe(Path(path))


def display_json(title: str, payload) -> None:
    display(Markdown(f"#### {title}"))
    if payload is None:
        print("<missing>")
        return
    print(json.dumps(payload, indent=2))


def preview_csv(path: Path, *, n: int = 10) -> pd.DataFrame | None:
    rows = preview_csv_rows(Path(path), n=n)
    if not rows:
        return None
    header = rows[0]
    body = rows[1:]
    if header and body:
        width = len(header)
        normalized_body = [row + ["" for _ in range(max(0, width - len(row)))] for row in body]
        df = pd.DataFrame(normalized_body, columns=header)
    else:
        df = pd.DataFrame(body)
    display(Markdown(f"#### CSV preview: `{path}`"))
    display(df.head(n))
    return df


## Framework health check (optional)

Run the canonical acceptance smoke command before notebook execution to verify the shipped assets and operator path.


In [ ]:
smoke_script = PROJECT_ROOT / "scripts" / "acceptance_smoke.py"
HEALTH_CHECK_OK = False

if not RUN_HEALTH_CHECK:
    HEALTH_CHECK_OK = True
    print("Skipping health check because RUN_HEALTH_CHECK=False")
else:
    if not smoke_script.exists():
        raise FileNotFoundError(f"Acceptance smoke script not found: {smoke_script}")

    health_check_result = run([sys.executable, str(smoke_script)], cwd=PROJECT_ROOT)
    HEALTH_CHECK_OK = health_check_result.returncode == 0
    if not HEALTH_CHECK_OK:
        raise RuntimeError(
            "Framework health check failed. Resolve acceptance smoke failures before continuing."
        )

print("Framework health check passed or intentionally skipped.")


## Inspect shipped registry and choose one experiment

Load the shipped registry, inspect available experiments, and validate the single selected experiment ID from the configuration cell.


In [ ]:
from Thesis_ML.demo.notebook_utils import select_registry_experiment

registry_payload = safe_read_json(REGISTRY_PATH)
if not isinstance(registry_payload, dict):
    raise RuntimeError(f"Invalid registry payload at: {REGISTRY_PATH}")

experiments = list(registry_payload.get("experiments", []))
if not experiments:
    raise RuntimeError(f"No experiments found in registry: {REGISTRY_PATH}")

registry_rows = []
for exp in experiments:
    templates = list(exp.get("variant_templates", []))
    supported_count = sum(1 for item in templates if bool(item.get("supported", True)))
    registry_rows.append(
        {
            "experiment_id": exp.get("experiment_id"),
            "title": exp.get("title"),
            "stage": exp.get("stage"),
            "primary_metric": exp.get("primary_metric"),
            "execution_status": exp.get("execution_status"),
            "executable_now": exp.get("executable_now"),
            "variant_templates": len(templates),
            "supported_templates": supported_count,
        }
    )

registry_table = pd.DataFrame(registry_rows).sort_values("experiment_id").reset_index(drop=True)
display(registry_table)
print(f"Registry schema_version: {registry_payload.get('schema_version')}")
print(f"Registry path: {REGISTRY_PATH}")

selected_experiment = select_registry_experiment(registry_payload, EXPERIMENT_ID)
print(f"Selected experiment ID: {EXPERIMENT_ID}")


## Create notebook-local single-experiment registry

To avoid accidental multi-experiment execution, write a notebook-local registry containing only the selected experiment.


In [ ]:
from Thesis_ML.demo.notebook_utils import create_single_experiment_registry

SINGLE_REGISTRY_PATH = NOTEBOOK_REGISTRY_DIR / f"single_experiment_{EXPERIMENT_ID}.json"
single_registry_payload = create_single_experiment_registry(
    source_registry_path=REGISTRY_PATH,
    destination_registry_path=SINGLE_REGISTRY_PATH,
    experiment_id=EXPERIMENT_ID,
)

print(f"Wrote single-experiment registry: {SINGLE_REGISTRY_PATH}")
display_json(
    "Single-experiment registry preview",
    {
        "schema_version": single_registry_payload.get("schema_version"),
        "description": single_registry_payload.get("description"),
        "experiment_ids": [
            exp.get("experiment_id") for exp in single_registry_payload.get("experiments", [])
        ],
    },
)


## Explain the selected experiment before execution

Review the experiment intent and key configuration fields so the run scope is explicit before launching commands.


In [ ]:
from Thesis_ML.demo.notebook_utils import summarize_registry_experiment

variant_templates = list(selected_experiment.get("variant_templates", []))
experiment_summary = summarize_registry_experiment(selected_experiment)

display(pd.DataFrame(experiment_summary.items(), columns=["field", "value"]))

template_rows = []
for template in variant_templates:
    params = template.get("params", {}) if isinstance(template.get("params"), dict) else {}
    template_rows.append(
        {
            "template_id": template.get("template_id"),
            "supported": template.get("supported", True),
            "start_section": template.get("start_section"),
            "end_section": template.get("end_section"),
            "base_artifact_id": template.get("base_artifact_id"),
            "target": params.get("target"),
            "model": params.get("model"),
            "cv": params.get("cv"),
            "subject": params.get("subject"),
            "train_subject": params.get("train_subject"),
            "test_subject": params.get("test_subject"),
            "filter_task": params.get("filter_task"),
            "filter_modality": params.get("filter_modality"),
        }
    )

display(pd.DataFrame(template_rows))


## Dry-run the selected experiment (default path)

Dry-run validates planning and orchestration safely without model fitting. This is the default demonstration path in this notebook.


In [ ]:
def build_campaign_command(*, dry_run: bool) -> list[str]:
    command = list(DECISION_SUPPORT_CMD) + [
        "--registry",
        str(SINGLE_REGISTRY_PATH),
        "--index-csv",
        str(INDEX_CSV),
        "--data-root",
        str(DATA_ROOT),
        "--cache-dir",
        str(CACHE_DIR),
        "--output-root",
        str(OUTPUT_ROOT),
        "--all",
    ]
    if dry_run:
        command.append("--dry-run")
    return command


DRY_RUN_OK = False
DRY_RUN_SKIPPED = False
DRY_RUN_CAMPAIGN_MANIFEST = None

prereq_issues = []
if not REGISTRY_PATH.exists():
    prereq_issues.append(f"Missing registry file: {REGISTRY_PATH}")
if not INDEX_CSV.exists():
    prereq_issues.append(f"Missing dataset index CSV: {INDEX_CSV}")
if not DATA_ROOT.exists():
    prereq_issues.append(f"Missing data root: {DATA_ROOT}")

if not RUN_DRY_RUN:
    DRY_RUN_SKIPPED = True
    print("Skipping dry-run because RUN_DRY_RUN=False")
elif prereq_issues:
    DRY_RUN_SKIPPED = True
    print("Skipping dry-run due to missing prerequisites:")
    for issue in prereq_issues:
        print("-", issue)
else:
    existing_before_dry_run = set(list_campaign_manifests(OUTPUT_ROOT))
    dry_run_command = build_campaign_command(dry_run=True)
    dry_run_result = run(dry_run_command, cwd=PROJECT_ROOT)
    DRY_RUN_OK = dry_run_result.returncode == 0

    after_dry_run = set(list_campaign_manifests(OUTPUT_ROOT))
    new_dry_run_manifests = sorted(
        [path for path in after_dry_run if path not in existing_before_dry_run],
        key=lambda p: p.stat().st_mtime,
    )
    if new_dry_run_manifests:
        DRY_RUN_CAMPAIGN_MANIFEST = new_dry_run_manifests[-1]
    elif after_dry_run:
        DRY_RUN_CAMPAIGN_MANIFEST = sorted(after_dry_run, key=lambda p: p.stat().st_mtime)[-1]

    print(f"Dry-run campaign manifest: {DRY_RUN_CAMPAIGN_MANIFEST}")
    if not DRY_RUN_OK:
        raise RuntimeError("Dry-run failed. Stop here and fix errors before full execution.")


## Optional full execution (guarded)

Full execution is optional and disabled by default (`RUN_FULL_EXECUTION=False`).
Enable it only when you have the required local data and want to run model fitting.


In [ ]:
FULL_RUN_OK = False
FULL_RUN_SKIPPED = False
FULL_RUN_CAMPAIGN_MANIFEST = None

if not RUN_FULL_EXECUTION:
    FULL_RUN_SKIPPED = True
    print("Skipping full execution because RUN_FULL_EXECUTION=False")
elif DRY_RUN_SKIPPED:
    FULL_RUN_SKIPPED = True
    print("Skipping full execution because dry-run was skipped.")
elif not DRY_RUN_OK:
    raise RuntimeError("Dry-run did not complete successfully; refusing full execution.")
else:
    existing_before_full_run = set(list_campaign_manifests(OUTPUT_ROOT))
    full_run_command = build_campaign_command(dry_run=False)
    full_run_result = run(full_run_command, cwd=PROJECT_ROOT)
    FULL_RUN_OK = full_run_result.returncode == 0

    after_full_run = set(list_campaign_manifests(OUTPUT_ROOT))
    new_full_run_manifests = sorted(
        [path for path in after_full_run if path not in existing_before_full_run],
        key=lambda p: p.stat().st_mtime,
    )
    if new_full_run_manifests:
        FULL_RUN_CAMPAIGN_MANIFEST = new_full_run_manifests[-1]
    elif after_full_run:
        FULL_RUN_CAMPAIGN_MANIFEST = sorted(after_full_run, key=lambda p: p.stat().st_mtime)[-1]

    print(f"Full-run campaign manifest: {FULL_RUN_CAMPAIGN_MANIFEST}")
    if not FULL_RUN_OK:
        raise RuntimeError("Full execution failed. Review command output above.")


## Inspect outputs on disk

Show a readable output tree and highlight important files.


In [ ]:
REFERENCE_CAMPAIGN_MANIFEST_PATH = FULL_RUN_CAMPAIGN_MANIFEST or DRY_RUN_CAMPAIGN_MANIFEST
CAMPAIGN_MANIFEST = (
    safe_read_json(REFERENCE_CAMPAIGN_MANIFEST_PATH)
    if REFERENCE_CAMPAIGN_MANIFEST_PATH is not None
    else None
)

if CAMPAIGN_MANIFEST:
    display_json("Selected campaign manifest", CAMPAIGN_MANIFEST)
else:
    print("No campaign manifest could be loaded.")

search_roots = [OUTPUT_ROOT]
if isinstance(CAMPAIGN_MANIFEST, dict):
    campaign_root_text = str(CAMPAIGN_MANIFEST.get("campaign_root") or "").strip()
    if campaign_root_text:
        search_roots.append(Path(campaign_root_text))
    experiment_roots = CAMPAIGN_MANIFEST.get("experiment_roots", {})
    if isinstance(experiment_roots, dict):
        for root_text in experiment_roots.values():
            root_text = str(root_text or "").strip()
            if root_text:
                search_roots.append(Path(root_text))

search_roots = dedupe_paths([path for path in search_roots if Path(path).exists()])
print("Search roots:")
for root in search_roots:
    print(f"- {root}")

KEY_FILENAMES = [
    "run_status.json",
    "config.json",
    "metrics.json",
    "predictions.csv",
    "fold_metrics.csv",
    "spatial_compatibility_report.json",
    "decision_support_summary.csv",
    "summary_outputs_export.csv",
    "run_log_export.csv",
    "artifact_registry.sqlite3",
]

KEY_FILE_MAP = {}
for name in KEY_FILENAMES:
    found_paths = []
    for root in search_roots:
        found_paths.extend(root.rglob(name))
    KEY_FILE_MAP[name] = dedupe_paths(found_paths)

key_rows = []
for name in KEY_FILENAMES:
    matches = KEY_FILE_MAP.get(name, [])
    key_rows.append(
        {
            "filename": name,
            "found_count": len(matches),
            "example_path": str(matches[0]) if matches else "",
        }
    )
display(pd.DataFrame(key_rows))

print_tree(OUTPUT_ROOT, max_depth=6, max_entries=500)


## Inspect key JSON outputs

Load and display run status, metrics, and spatial compatibility report where present.


In [ ]:
JSON_PRIORITY = [
    "run_status.json",
    "metrics.json",
    "spatial_compatibility_report.json",
]

for filename in JSON_PRIORITY:
    candidates = KEY_FILE_MAP.get(filename, [])
    if not candidates:
        print(f"[not found] {filename}")
        continue
    payload = safe_read_json(candidates[0])
    display_json(f"{filename} ({candidates[0]})", payload)


## Inspect key CSV outputs

Preview predictions, fold metrics, and summary CSVs when available.


In [ ]:
CSV_PRIORITY = [
    "predictions.csv",
    "fold_metrics.csv",
    "decision_support_summary.csv",
    "summary_outputs_export.csv",
    "run_log_export.csv",
]

for filename in CSV_PRIORITY:
    candidates = KEY_FILE_MAP.get(filename, [])
    if not candidates:
        print(f"[not found] {filename}")
        continue
    preview_csv(candidates[0], n=8)


## Inspect artifact lineage (SQLite)

Artifact lineage matters for provenance and auditability: it records what artifacts were produced, their run IDs, and upstream dependencies used to create them.


In [ ]:
artifact_registry_candidates = list(KEY_FILE_MAP.get("artifact_registry.sqlite3", []))
if not artifact_registry_candidates and isinstance(CAMPAIGN_MANIFEST, dict):
    registry_path_text = str(CAMPAIGN_MANIFEST.get("artifact_registry_path") or "").strip()
    if registry_path_text:
        artifact_registry_candidates = [Path(registry_path_text)]

ARTIFACT_REGISTRY_PATH = artifact_registry_candidates[0] if artifact_registry_candidates else None
if ARTIFACT_REGISTRY_PATH is None or not Path(ARTIFACT_REGISTRY_PATH).exists():
    print("Artifact registry SQLite file not found for this run.")
else:
    ARTIFACT_REGISTRY_PATH = Path(ARTIFACT_REGISTRY_PATH)
    print(f"artifact_registry_path={ARTIFACT_REGISTRY_PATH}")
    with sqlite3.connect(str(ARTIFACT_REGISTRY_PATH)) as connection:
        tables_df = pd.read_sql_query(
            "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
            connection,
        )
        display(Markdown("#### SQLite tables"))
        display(tables_df)

        table_names = set(tables_df["name"].astype(str).tolist())
        if "artifacts" in table_names:
            artifacts_df = pd.read_sql_query(
                """
                SELECT artifact_id, artifact_type, run_id, upstream_artifact_ids,
                       status, created_at, path
                FROM artifacts
                ORDER BY created_at DESC
                LIMIT 30
                """,
                connection,
            )
            display(Markdown("#### artifacts table preview"))
            display(artifacts_df)

            lineage_rows = []
            for _, row in artifacts_df.iterrows():
                raw_upstream = row.get("upstream_artifact_ids")
                upstream_ids = []
                if isinstance(raw_upstream, str) and raw_upstream.strip():
                    try:
                        parsed = json.loads(raw_upstream)
                        if isinstance(parsed, list):
                            upstream_ids = [str(item) for item in parsed]
                    except Exception:
                        upstream_ids = []

                if not upstream_ids:
                    lineage_rows.append(
                        {
                            "artifact_id": row.get("artifact_id"),
                            "upstream_artifact_id": None,
                            "artifact_type": row.get("artifact_type"),
                            "status": row.get("status"),
                        }
                    )
                    continue

                for upstream_id in upstream_ids:
                    lineage_rows.append(
                        {
                            "artifact_id": row.get("artifact_id"),
                            "upstream_artifact_id": upstream_id,
                            "artifact_type": row.get("artifact_type"),
                            "status": row.get("status"),
                        }
                    )

            lineage_df = pd.DataFrame(lineage_rows)
            display(Markdown("#### Lineage edges (artifact -> upstream artifact)"))
            display(lineage_df.head(40))
        else:
            print("No 'artifacts' table found in artifact registry database.")


## Final run summary

Build a compact summary with status, important file counts, and key artifact paths.


In [ ]:
def first_path(filename: str) -> str:
    candidates = KEY_FILE_MAP.get(filename, [])
    return str(candidates[0]) if candidates else ""


important_unique_paths = sorted(
    {str(path) for paths in KEY_FILE_MAP.values() for path in paths}
)

if DRY_RUN_SKIPPED:
    dry_run_status = "skipped"
elif DRY_RUN_OK:
    dry_run_status = "succeeded"
else:
    dry_run_status = "failed"

if FULL_RUN_SKIPPED:
    full_run_status = "skipped"
elif FULL_RUN_OK:
    full_run_status = "succeeded"
else:
    full_run_status = "failed"

FINAL_SUMMARY = {
    "selected_experiment_id": EXPERIMENT_ID,
    "registry_path_used": str(SINGLE_REGISTRY_PATH),
    "output_root": str(OUTPUT_ROOT),
    "run_health_check": bool(RUN_HEALTH_CHECK),
    "run_dry_run": bool(RUN_DRY_RUN),
    "run_full_execution": bool(RUN_FULL_EXECUTION),
    "dry_run_status": dry_run_status,
    "full_execution_status": full_run_status,
    "important_files_found_count": int(len(important_unique_paths)),
    "key_result_paths": {
        "campaign_manifest": (
            str(REFERENCE_CAMPAIGN_MANIFEST_PATH) if REFERENCE_CAMPAIGN_MANIFEST_PATH else ""
        ),
        "run_status": first_path("run_status.json"),
        "config": first_path("config.json"),
        "metrics": first_path("metrics.json"),
        "predictions": first_path("predictions.csv"),
        "fold_metrics": first_path("fold_metrics.csv"),
        "spatial_compatibility_report": first_path("spatial_compatibility_report.json"),
        "decision_support_summary": first_path("decision_support_summary.csv"),
        "artifact_registry": first_path("artifact_registry.sqlite3"),
    },
}

display(pd.DataFrame(FINAL_SUMMARY.items(), columns=["field", "value"]))
display_json("Final summary (JSON)", FINAL_SUMMARY)


## Interpretation and limits

### What this notebook demonstrates
- Portable, registry-driven execution setup for one predefined experiment.
- Safe planning via dry-run as the default first-step path.
- Optional guarded full execution through the same CLI interface.
- Artifact/output/provenance inspection from generated campaign files.

### What this notebook does not establish scientifically
- It does **not** prove scientific validity, causal inference, or statistical significance.
- It does **not** replace protocol design, dataset quality checks, or confirmatory analysis plans.
- It demonstrates workflow mechanics and reproducible execution plumbing only.

### Next step
Proceed to `02_single_experiment_workbook_walkthrough.ipynb` for workbook-driven execution flow.
